In [34]:
import pandas as pd
import numpy as np

In [35]:
file_path = r"C:\Users\Utkarsh\Desktop\Customer Churn Analysis and Prediction Pipeline\Customer_Data.csv"
data = pd.read_csv(file_path)

In [36]:
data.head()

,Customer_ID,Gender,Age,Married,State,Number_of_Referrals,Tenure_in_Months,Value_Deal,Phone_Service,Multiple_Lines,...,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Churn_Category,Churn_Reason
0,19877-DEL,Male,35,No,Delhi,7,27,NaN,Yes,No,...,Credit Card,65.6,593.30,0.0,0,381.51,974.81,Stayed,NaN,NaN
1,25063-WES,Male,51,No,West Bengal,4,35,Deal 5,Yes,No,...,Bank Withdrawal,73.9,280.85,0.0,0,134.60,415.45,Churned,Competitor,Competitor had better devices
2,59787-KAR,Male,79,No,Karnataka,3,21,Deal 4,Yes,No,...,Bank Withdrawal,98.0,1237.85,0.0,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
3,28544-TAM,Female,80,No,Tamil Nadu,3,8,NaN,Yes,No,...,Credit Card,83.9,267.40,0.0,0,22.14,289.54,Churned,Dissatisfaction,Network reliability
4,32750-TEL,Male,24,No,Telangana,9,2,Deal 5,Yes,No,...,Credit Card,69.4,571.45,0.0,0,150.93,722.38,Stayed,NaN,NaN


In [37]:
# 1. Rows to change: state is Uttar Pradesh AND the ID currently ends in -UTT
mask = (data["State"] == "Uttar Pradesh") & data["Customer_ID"].str.endswith("-UTT")

# 2. Replace the last 3 characters "UTT" with "UT", in place
data.loc[mask, "Customer_ID"] = data.loc[mask, "Customer_ID"].str[:-3] + "UT"

In [38]:
# Check for columns with null/blank values
cols_with_nulls = data.columns[data.isna().any()].tolist()
print(cols_with_nulls)


['Value_Deal', 'Multiple_Lines', 'Internet_Type', 'Online_Security', 'Online_Backup', 'Device_Protection_Plan', 'Premium_Support', 'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data', 'Churn_Category', 'Churn_Reason']


In [39]:
# Value_Deal: no deal taken
data['Value_Deal'] = data['Value_Deal'].fillna('No Deal')

data['Internet_Type'] = data['Internet_Type'].fillna('No Internet Type')

# Internet-related columns: null only when the customer has no internet service
internet_cols = ['Multiple_Lines', 'Online_Security', 'Online_Backup',
                 'Device_Protection_Plan', 'Premium_Support', 'Streaming_TV',
                 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data']
data[internet_cols] = data[internet_cols].fillna('No')


In [40]:
# Checking customers
data.loc[data['Customer_Status'] == 'Stayed', ['Customer_Status', 'Churn_Category', 'Churn_Reason']]

,Customer_Status,Churn_Category,Churn_Reason
0,Stayed,NaN,NaN
4,Stayed,NaN,NaN
5,Stayed,NaN,NaN
6,Stayed,NaN,NaN
7,Stayed,NaN,NaN
...,...,...,...
6303,Stayed,NaN,NaN
6304,Stayed,NaN,NaN
6307,Stayed,NaN,NaN
6309,Stayed,NaN,NaN


In [41]:
stayed = data['Customer_Status'] == 'Stayed'
print(data.loc[stayed, 'Churn_Category'].notna().sum())
print(data.loc[stayed, 'Churn_Reason'].notna().sum())

0
0


In [42]:
data.loc[stayed & data['Churn_Category'].isna(), 'Churn_Category'] = 'Not Churned'
data.loc[stayed & data['Churn_Reason'].isna(), 'Churn_Reason'] = 'Not Churned'

In [43]:
joined = data['Customer_Status'] == 'Joined'
print(data.loc[joined, 'Churn_Category'].notna().sum())
print(data.loc[joined, 'Churn_Reason'].notna().sum())


0
0


In [44]:
data.loc[joined & data['Churn_Category'].isna(), 'Churn_Category'] = 'N/A - Joined'
data.loc[joined & data['Churn_Reason'].isna(), 'Churn_Reason'] = 'N/A - Joined'

In [45]:
churned = data['Customer_Status'] == 'Churned'
print(data.loc[churned, 'Churn_Category'].isna().sum())
print(data.loc[churned, 'Churn_Reason'].isna().sum())

0
0


In [46]:


data['Calc_Total'] = data['Tenure_in_Months'] * data['Monthly_Charge']
match = np.isclose(data['Calc_Total'], data['Total_Charges'], atol=0.01)

print("Matches:", match.sum(), "of", len(data))
print("Mismatches:", (~match).sum())

data = data.drop(columns='Calc_Total')

Matches: 18 of 6311
Mismatches: 6293


In [47]:
# Check the tenue for Joined Customers

joined = data['Customer_Status'] == 'Joined'
print(data.loc[joined, 'Tenure_in_Months'].value_counts().sort_index())
print(data.loc[joined, 'Tenure_in_Months'].max())

Tenure_in_Months
1     11
2     16
3     13
4     15
5     12
6     12
7     17
8     15
9     18
10    21
11    12
12     9
13    15
14    14
15     8
16    12
17     4
18    15
19    12
20    10
21     9
22    11
23    10
24     7
25    10
26     3
27     8
28     9
29    12
30    11
31     9
32     8
33    13
34    11
35     6
36     7
Name: count, dtype: int64
36


In [48]:
data['Tenure_in_Months'] = (data['Total_Charges'] / data['Monthly_Charge']).round().astype(int)

In [49]:
churned_stayed = data[data['Customer_Status'].isin(['Churned', 'Stayed'])]
joined = data[data['Customer_Status'] == 'Joined']

# Write both to one Excel file, one sheet each
with pd.ExcelWriter('Customer_Data_Split.xlsx', engine='openpyxl') as writer:
    churned_stayed.to_excel(writer, sheet_name='Churned_Stayed', index=False)
    joined.to_excel(writer, sheet_name='Joined', index=False)